# Sentiment Analysis for Customer Feedback – Product and Service Improvements with Precision

- ##### Company Overview 

ShopEase Europe is a fast-growing e-commerce company headquartered in London, operating across the United Kingdom, France, Germany, and Spain. The company serves over one million monthly users and focuses on delivering seamless digital shopping experiences through innovation, personalization, and customer trust. As the customer base grows, the company receives large volumes of feedback from reviews, social media, and support channels.

- #####  Business Challenge

ShopEase Europe faces increasing difficulty in manually analysing customer feedback due to:

High Feedback Volume – Thousands of customer reviews are generated daily, making manual analysis inefficient.
Subjective Interpretation – Human analysts often interpret sentiment differently, leading to inconsistent results.
Limited Actionable Insights – Extracting useful business recommendations from unstructured text is slow and fragmented.

The company needs an automated system capable of accurately analysing customer sentiment at scale.

- ##### Project Objectives

The project aims to:

Build a multilingual text-processing pipeline for customer feedback.
Develop sentiment classification models to categorise feedback as positive, negative, or neutral.
Identify major drivers of customer satisfaction, such as delivery speed, product quality, pricing, and customer support.
Generate actionable insights and recommendations for business teams.
Create an interactive dashboard for real-time sentiment monitoring.
- ##### Project Methodology

Step 1: Data Collection & Cleaning
Collect customer reviews from e-commerce platforms, social media, and support channels. Remove duplicates, handle missing data, and normalize text.

Step 2: Data Preprocessing
Perform tokenisation, lemmatisation, text vectorisation using TF-IDF and contextual embeddings, and balance the dataset.

Step 3: Exploratory Data Analysis (EDA)
Analyse sentiment patterns across countries and product categories, and identify frequently used keywords.

Step 4: Model Development
Train traditional models such as Naïve Bayes and Logistic Regression, then advanced NLP models like BERT for improved classification performance.

Step 5: Insight Generation
Extract sentiment drivers, summarise customer opinions, and identify recurring complaints or satisfaction trends.

Step 6: Deployment & Monitoring
Deploy the model using Streamlit or Flask and continuously monitor model performance for drift.

- #####  Expected Outcome

The final solution will provide a scalable sentiment analysis system capable of:

Delivering real-time customer sentiment insights.
Detecting major factors affecting customer experience.
Supporting faster decision-making for marketing, product, and operations teams.
Improving customer satisfaction, loyalty, and overall competitiveness in the European e-commerce market.

This project will enable ShopEase Europe to move from manual feedback analysis to a data-driven, automated customer intelligence system that strengthens long-term business performance.

# Import Required Libraries

In [1]:
# EXPLAINABILITY NOTE:
# This cell imports all libraries used later in the workflow.
# Optional packages are wrapped in fallback logic so the notebook can still run
# when language-detection or transformer libraries are not installed.

# Import type hints for better code documentation and IDE support
from typing import Optional, List, Tuple, Any
import os
import re
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# Import the datetime class from the datetime module for working with dates and times
from datetime import datetime

# Optional text-repair dependency. The fallback keeps the notebook runnable.
try:
    import ftfy
except ImportError:
    class ftfy:
        @staticmethod
        def fix_text(text):
            return text

# Optional language detection dependency. Missing package should not stop modelling.
try:
    from langdetect import detect, DetectorFactory
    from langdetect.lang_detect_exception import LangDetectException
    DetectorFactory.seed = 0
except ImportError:
    class LangDetectException(Exception):
        pass

    def detect(text):
        return "unknown"

# NLP preprocessing libraries
import spacy
import nltk
from nltk.corpus import stopwords

# Classical ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

# Deep learning libraries
import torch
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Hugging Face libraries are optional because they may need installation/downloads.
try:
    from datasets import Dataset
    from transformers import (
        DistilBertTokenizerFast,
        DistilBertForSequenceClassification,
        Trainer,
        TrainingArguments,
    )
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False

# LIME explains individual predictions by showing which words
# most influenced the model’s decision (positive or negative impact)
from lime.lime_text import LimeTextExplainer

# defaultdict stores values for words automatically without manual initialisation
# Counter helps count and rank how often words influence predictions
from collections import defaultdict, Counter

# Import joblib library for loading/saving machine learning models and data
import joblib
# Import Streamlit library for creating web applications
import streamlit as st


# Import Dataset From working Directory

In [2]:
# EXPLAINABILITY NOTE:
# This cell loads the raw customer review CSV into a DataFrame.
# All later cleaning, exploratory analysis, and modelling steps start from this data.

# Load review data from CSV file into a pandas DataFrame
df = pd.read_csv("processed_sentiment_data.csv")
# Display the first 5 rows of the DataFrame to inspect the data structure
df.head()

,review_id,product_category,timestamp,country,rating,review,sentiment,_clean_text,detected_language,is_valid_detection,processed_review
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a ...",negative,i registered on the website tried to order a l...,en,True,registered website tried order laptop entered ...
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,Had multiple orders one turned up and driver h...,negative,had multiple orders one turned up and driver h...,en,True,multiple orders one turned driver phone door n...
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,I informed these reprobates that I WOULD NOT B...,negative,i informed these reprobates that i would not b...,en,True,informed reprobates would going visit sick rel...
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no proble...,negative,i have bought from amazon before and no proble...,en,True,bought amazon problems happy service price ama...
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancel...,negative,if i could give a lower rate i would i cancell...,en,True,could give lower rate would cancelled amazon p...


# Classical NLP Modeling

In [15]:
# EXPLAINABILITY NOTE:
# This cell trains classical NLP models with safeguards against overfitting.
# It removes duplicate review/label pairs before splitting, fits TF-IDF only on training data,
# uses stratified validation, and reports train-vs-test gaps for memorisation checks.

def auto_detect_text_column_safe(dataframe):
    """Find the most likely review text column for analysis/modelling cells."""
    possible_columns = ["processed_review", "review_text", "review", "text", "content", "comment", "message"]

    for col in possible_columns:
        if col in dataframe.columns:
            return col

    raise ValueError(f"No valid text column found. Available columns: {list(dataframe.columns)}")


# ============================================================
# CLASSICAL NLP MODELS WITH LEAKAGE AND OVERFITTING CONTROLS
# ============================================================

def prepare_sentiment_model_data(df, text_column=None, label_column="sentiment"):
    if text_column is None:
        if "processed_review" in df.columns and df["processed_review"].astype(str).str.strip().ne("").any():
            text_column = "processed_review"
        else:
            text_column = auto_detect_text_column_safe(df)

    model_df = df[[text_column, label_column]].copy()
    model_df.columns = ["text", "label"]
    model_df["text"] = model_df["text"].astype(str).str.strip()
    model_df["label"] = model_df["label"].astype(str).str.lower().str.strip()
    model_df = model_df[(model_df["text"] != "") & (model_df["label"] != "")]

    # Remove exact duplicate text/label rows. This prevents the same review
    # appearing in both train and test sets, which creates over-optimistic scores.
    before = len(model_df)
    model_df = model_df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
    duplicates_removed = before - len(model_df)

    # If the same text has conflicting labels, remove those ambiguous rows.
    label_counts = model_df.groupby("text")["label"].nunique()
    conflicting_texts = label_counts[label_counts > 1].index
    conflict_count = len(conflicting_texts)
    if conflict_count:
        model_df = model_df[~model_df["text"].isin(conflicting_texts)].reset_index(drop=True)

    print("Rows before duplicate removal:", before)
    print("Duplicate text/label rows removed:", duplicates_removed)
    print("Conflicting duplicate texts removed:", conflict_count)
    print("Rows used for modelling:", len(model_df))
    print("\nClass distribution after de-duplication")
    print(model_df["label"].value_counts())

    return model_df


model_df = prepare_sentiment_model_data(df)

X = model_df["text"]
y = model_df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True,
)

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", tfidf),
        ("classifier", LogisticRegression(
            C=0.35,
            penalty="l2",
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "Complement Naive Bayes": Pipeline([
        ("tfidf", tfidf),
        ("classifier", ComplementNB(alpha=1.0)),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", tfidf),
        ("classifier", LinearSVC(
            C=0.35,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}

for name, model in models.items():
    print("\n========================================")
    print(name.upper())
    print("========================================")

    cv_scores = cross_val_score(
        model,
        X_train_text,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
    )

    model.fit(X_train_text, y_train)
    train_predictions = model.predict(X_train_text)
    test_predictions = model.predict(X_test_text)

    train_f1 = f1_score(y_train, train_predictions, average="macro")
    test_f1 = f1_score(y_test, test_predictions, average="macro")
    test_accuracy = accuracy_score(y_test, test_predictions)
    overfit_gap = train_f1 - test_f1

    print("CV Macro F1:", round(cv_scores.mean(), 4), "+/-", round(cv_scores.std(), 4))
    print("Train Macro F1:", round(train_f1, 4))
    print("Test Macro F1:", round(test_f1, 4))
    print("Overfit Gap:", round(overfit_gap, 4))
    print("Test Accuracy:", round(test_accuracy, 4))
    print(classification_report(y_test, test_predictions, zero_division=0))
    print(confusion_matrix(y_test, test_predictions))

    results.append({
        "Model": name,
        "CV_Macro_F1": cv_scores.mean(),
        "CV_Std": cv_scores.std(),
        "Train_Macro_F1": train_f1,
        "Test_Macro_F1": test_f1,
        "Overfit_Gap": overfit_gap,
        "Test_Accuracy": test_accuracy,
    })
    trained_models[name] = model

comparison = pd.DataFrame(results).sort_values(
    ["Test_Macro_F1", "Overfit_Gap"],
    ascending=[False, True],
)

print("\nMODEL PERFORMANCE COMPARISON")
print(comparison)

best_classical_model_name = comparison.iloc[0]["Model"]
best_classical_model = trained_models[best_classical_model_name]

print("\nBEST CLASSICAL MODEL")
print(comparison.iloc[0])

Rows before duplicate removal: 21054
Duplicate text/label rows removed: 672
Conflicting duplicate texts removed: 8
Rows used for modelling: 20365

Class distribution after de-duplication
label
negative    14149
positive     5395
neutral       821
Name: count, dtype: int64

LOGISTIC REGRESSION
CV Macro F1: 0.6509 +/- 0.0065
Train Macro F1: 0.7742
Test Macro F1: 0.6532
Overfit Gap: 0.121
Test Accuracy: 0.8407
              precision    recall  f1-score   support

    negative       0.95      0.87      0.91      2830
     neutral       0.15      0.38      0.21       164
    positive       0.85      0.82      0.83      1079

    accuracy                           0.84      4073
   macro avg       0.65      0.69      0.65      4073
weighted avg       0.89      0.84      0.86      4073

[[2472  242  116]
 [  55   63   46]
 [  69  121  889]]

COMPLEMENT NAIVE BAYES
CV Macro F1: 0.5982 +/- 0.0022
Train Macro F1: 0.6302
Test Macro F1: 0.5974
Overfit Gap: 0.0329
Test Accuracy: 0.8807
           

### Saved Classical NLP Models

In [16]:
# ============================================================
# SAVE TRAINED CLASSICAL NLP MODELS
#
# Purpose:
# Save each trained model after training so they can be
# loaded later for deployment or future prediction.
#
# Why:
# Training models repeatedly is computationally expensive.
# Saving allows direct reuse without retraining.
#
# Saved files:
# 1. trained_log_reg_model.pkl
# 2. trained_NB_model.pkl
# 3. trained_SVM_model.pkl
# ============================================================

import joblib


# ============================================================
# SAVE LOGISTIC REGRESSION MODEL
#
# Purpose:
# Save trained Logistic Regression pipeline
#
# Contains:
# TF-IDF Vectorizer + Logistic Regression classifier
# ============================================================

joblib.dump(

    trained_models["Logistic Regression"],

    "trained_log_reg_model.pkl"
)

print("Logistic Regression model saved successfully")


# ============================================================
# SAVE NAIVE BAYES MODEL
#
# Purpose:
# Save trained Complement Naive Bayes pipeline
#
# Contains:
# TF-IDF Vectorizer + Naive Bayes classifier
# ============================================================

joblib.dump(

    trained_models["Complement Naive Bayes"],

    "trained_NB_model.pkl"
)

print("Naive Bayes model saved successfully")


# ============================================================
# SAVE LINEAR SVM MODEL
#
# Purpose:
# Save trained Support Vector Machine pipeline
#
# Contains:
# TF-IDF Vectorizer + Linear SVM classifier
# ============================================================

joblib.dump(

    trained_models["Linear SVM"],

    "trained_SVM_model.pkl"
)

print("Linear SVM model saved successfully")


# ============================================================
# FINAL CONFIRMATION
#
# Purpose:
# Confirm all trained models were saved correctly
#
# Why:
# Saved models can now be loaded directly into Streamlit
# deployment without retraining.
# ============================================================

print("\n=======================================")
print("ALL MODELS SAVED SUCCESSFULLY")
print("trained_log_reg_model.pkl")
print("trained_NB_model.pkl")
print("trained_SVM_model.pkl")
print("Ready for deployment")
print("=======================================")

Logistic Regression model saved successfully
Naive Bayes model saved successfully
Linear SVM model saved successfully

ALL MODELS SAVED SUCCESSFULLY
trained_log_reg_model.pkl
trained_NB_model.pkl
trained_SVM_model.pkl
Ready for deployment


# Transformer Model Development

In [14]:
# EXPLAINABILITY NOTE:
# This cell compares different regularisation strengths for linear models.
# The overfit gap shows whether higher training performance is failing to generalise.

# ============================================================
# CLASSICAL MODEL REGULARIZATION CHECK
# Purpose: verify that stronger models are not simply memorising reviews
# ============================================================

regularization_rows = []

for c_value in [0.1, 0.25, 0.5, 1.0]:
    for model_name, classifier in {
        "Logistic Regression": LogisticRegression(
            C=c_value,
            penalty="l2",
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        ),
        "Linear SVM": LinearSVC(
            C=c_value,
            class_weight="balanced",
            random_state=42,
        ),
    }.items():
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=5000,
                stop_words="english",
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.85,
                sublinear_tf=True,
            )),
            ("classifier", classifier),
        ])

        pipeline.fit(X_train_text, y_train)
        train_pred = pipeline.predict(X_train_text)
        test_pred = pipeline.predict(X_test_text)

        regularization_rows.append({
            "Model": model_name,
            "C": c_value,
            "Train_Macro_F1": f1_score(y_train, train_pred, average="macro"),
            "Test_Macro_F1": f1_score(y_test, test_pred, average="macro"),
            "Overfit_Gap": (
                f1_score(y_train, train_pred, average="macro")
                - f1_score(y_test, test_pred, average="macro")
            ),
        })

regularization_report = pd.DataFrame(regularization_rows).sort_values(
    ["Test_Macro_F1", "Overfit_Gap"],
    ascending=[False, True],
)

print("REGULARIZATION REPORT")
print(regularization_report)

REGULARIZATION REPORT
                 Model     C  Train_Macro_F1  Test_Macro_F1  Overfit_Gap
3           Linear SVM  0.25        0.878376       0.663394     0.214981
5           Linear SVM  0.50        0.913062       0.662574     0.250488
7           Linear SVM  1.00        0.936290       0.659186     0.277103
1           Linear SVM  0.10        0.826135       0.653788     0.172347
6  Logistic Regression  1.00        0.820393       0.651133     0.169260
4  Logistic Regression  0.50        0.787832       0.650643     0.137189
2  Logistic Regression  0.25        0.755179       0.644395     0.110785
0  Logistic Regression  0.10        0.716348       0.638736     0.077612


In [15]:
# EXPLAINABILITY NOTE:
# This cell trains deep learning and transformer models with overfitting controls.
# It uses a validation split, dropout, class weights, early stopping, learning-rate reduction,
# and safe transformer fallback logic when Hugging Face dependencies are unavailable.

# ============================================================
# DEEP LEARNING AND TRANSFORMER MODELS WITH OVERFITTING CONTROLS
# ============================================================

# Use raw/deduplicated text splits from the classical modelling cell.
if "X_train_text" not in globals() or "X_test_text" not in globals():
    model_df = prepare_sentiment_model_data(df)
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        model_df["text"],
        model_df["label"],
        test_size=0.2,
        stratify=model_df["label"],
        random_state=42,
    )

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_train_text,
    y_train_encoded,
    test_size=0.2,
    stratify=y_train_encoded,
    random_state=42,
)

# ============================================================
# BILSTM MODEL
# ============================================================

max_words = 10000
max_len = 80

tokenizer = Tokenizer(
    num_words=max_words,
    oov_token="<OOV>",
)
tokenizer.fit_on_texts(X_train_dl)

X_train_seq = tokenizer.texts_to_sequences(X_train_dl)
X_val_seq = tokenizer.texts_to_sequences(X_val_dl)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_encoded),
    y=y_train_encoded,
)
class_weights = dict(enumerate(class_weights_array))

bilstm_model = Sequential([
    Embedding(
        input_dim=max_words,
        output_dim=64,
        embeddings_regularizer=l2(1e-5),
    ),
    SpatialDropout1D(0.30),
    Bidirectional(LSTM(
        32,
        dropout=0.30,
        recurrent_dropout=0.20,
        kernel_regularizer=l2(1e-4),
    )),
    Dropout(0.50),
    Dense(
        len(label_encoder.classes_),
        activation="softmax",
        kernel_regularizer=l2(1e-4),
    ),
])

bilstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(
        monitor= "val_loss",
        patience=2,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor= "val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
    ),
]

history = bilstm_model.fit(
    X_train_pad,
    y_train_dl,
    epochs=10,
    batch_size=64,
    validation_data=(X_val_pad, y_val_dl),
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

bilstm_probabilities = bilstm_model.predict(X_test_pad)
bilstm_pred = np.argmax(bilstm_probabilities, axis=1)

bilstm_accuracy = accuracy_score(y_test_encoded, bilstm_pred)
bilstm_macro_f1 = f1_score(y_test_encoded, bilstm_pred, average="macro")

print("\nBILSTM TEST RESULTS")
print("Accuracy:", round(bilstm_accuracy, 4))
print("Macro F1:", round(bilstm_macro_f1, 4))
print(classification_report(y_test_encoded, bilstm_pred, target_names=label_encoder.classes_, zero_division=0))


# ============================================================
# DISTILBERT TRANSFORMER MODEL
# ============================================================

distilbert_accuracy = np.nan
distilbert_macro_f1 = np.nan

if not HF_AVAILABLE:
    print("\nSkipping DistilBERT: install 'transformers' and 'datasets' to run this section.")
else:
    try:
        hf_train_df = pd.DataFrame({
            "text": X_train_dl.astype(str).tolist(),
            "label": y_train_dl.tolist(),
        })
        hf_val_df = pd.DataFrame({
            "text": X_val_dl.astype(str).tolist(),
            "label": y_val_dl.tolist(),
        })
        hf_test_df = pd.DataFrame({
            "text": X_test_text.astype(str).tolist(),
            "label": y_test_encoded.tolist(),
        })

        train_dataset = Dataset.from_pandas(hf_train_df, preserve_index=False)
        val_dataset = Dataset.from_pandas(hf_val_df, preserve_index=False)
        test_dataset = Dataset.from_pandas(hf_test_df, preserve_index=False)

        bert_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

        def tokenize(batch):
            return bert_tokenizer(
                batch["text"],
                padding="max_length",
                truncation=True,
                max_length=96,
            )

        train_dataset = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
        val_dataset = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
        test_dataset = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

        model = DistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            num_labels=len(label_encoder.classes_),
            seq_classif_dropout=0.30,
        )

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=1)
            return {
                "accuracy": accuracy_score(labels, preds),
                "f1_macro": f1_score(labels, preds, average="macro"),
            }

        training_config = dict(
            output_dir="./results",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            weight_decay=0.01,
            warmup_ratio=0.10,
            save_strategy="epoch",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            save_total_limit=1,
            report_to="none",
        )

        try:
            training_args = TrainingArguments(eval_strategy="epoch", **training_config)
        except TypeError:
            training_args = TrainingArguments(evaluation_strategy="epoch", **training_config)

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        pred = trainer.predict(test_dataset)
        distilbert_pred = np.argmax(pred.predictions, axis=1)

        distilbert_accuracy = accuracy_score(y_test_encoded, distilbert_pred)
        distilbert_macro_f1 = f1_score(y_test_encoded, distilbert_pred, average="macro")

        print("\nDISTILBERT TEST RESULTS")
        print("Accuracy:", round(distilbert_accuracy, 4))
        print("Macro F1:", round(distilbert_macro_f1, 4))
        print(classification_report(y_test_encoded, distilbert_pred, target_names=label_encoder.classes_, zero_division=0))

    except Exception as exc:
        print("\nSkipping DistilBERT because the transformer setup failed:")
        print(exc)


comparison = pd.DataFrame({
    "Model": ["BiLSTM", "DistilBERT"],
    "Accuracy": [bilstm_accuracy, distilbert_accuracy],
    "Macro_F1": [bilstm_macro_f1, distilbert_macro_f1],
}).dropna()

print("\nFINAL DEEP / TRANSFORMER MODEL COMPARISON")
print(comparison)

if not comparison.empty:
    best_model = comparison.loc[comparison["Macro_F1"].idxmax()]
    print("\nBEST FINAL MODEL")
    print(best_model)

Epoch 1/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 57s 194ms/step - accuracy: 0.7140 - loss: 0.9251 - val_accuracy: 0.7949 - val_loss: 0.6824 - learning_rate: 0.0010
Epoch 2/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 28s 137ms/step - accuracy: 0.8033 - loss: 0.7483 - val_accuracy: 0.8462 - val_loss: 0.4698 - learning_rate: 0.0010
Epoch 3/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 27s 131ms/step - accuracy: 0.8048 - loss: 0.6674 - val_accuracy: 0.8103 - val_loss: 0.5070 - learning_rate: 0.0010
Epoch 4/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 29s 140ms/step - accuracy: 0.8271 - loss: 0.5796 - val_accuracy: 0.8179 - val_loss: 0.4862 - learning_rate: 5.0000e-04
128/128 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step

BILSTM TEST RESULTS
Accuracy: 0.8404
Macro F1: 0.6287
              precision    recall  f1-score   support

    negative       0.94      0.89      0.91      2830
     neutral       0.10      0.24      0.14       164
    positive       0.87      0.80      0.83      1078

    accuracy                           0.84      4072
   macr

Map:   0%|          | 0/13027 [00:00<?, ? examples/s]

Map:   0%|          | 0/3257 [00:00<?, ? examples/s]

Map:   0%|          | 0/4072 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.335166,0.325247,0.909733,0.612478
2,0.316302,0.302887,0.907277,0.633880
3,0.135150,0.308258,0.918023,0.653443


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DISTILBERT TEST RESULTS
Accuracy: 0.9126
Macro F1: 0.6758
              precision    recall  f1-score   support

    negative       0.94      0.96      0.95      2830
     neutral       0.31      0.13      0.19       164
    positive       0.88      0.90      0.89      1078

    accuracy                           0.91      4072
   macro avg       0.71      0.66      0.68      4072
weighted avg       0.90      0.91      0.90      4072


FINAL DEEP / TRANSFORMER MODEL COMPARISON
        Model  Accuracy  Macro_F1
0      BiLSTM  0.840373  0.628747
1  DistilBERT  0.912574  0.675835

BEST FINAL MODEL
Model       DistilBERT
Accuracy      0.912574
Macro_F1      0.675835
Name: 1, dtype: object


In [16]:
# ============================================================
# SAVE TRAINED DISTILBERT MODEL
# Purpose: Save final trained model for later deployment
# ============================================================

model.save_pretrained("./final_distilbert_model")

bert_tokenizer.save_pretrained("./final_distilbert_model")

print("Model saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully
